# T2I-RL 快速训练 (SiliconFlow 免费 VLM)

**使用国产免费 API 进行 VLM Reward 训练**

## 优势
- **超低成本**: SiliconFlow 提供极低价格的 VLM API
- **国内访问快**: 无需翻墙
- **兼容 OpenAI 格式**: 代码改动最小

## 训练成本估算
- 100 prompts × 4 images × 5 epochs = 2000 次调用
- 约 ¥0.4 总成本 (Qwen2.5-VL-32B)

---

## Step 0: 配置 API Key

**获取 API Key**: https://cloud.siliconflow.cn/

1. 注册账号
2. 进入控制台
3. 创建 API Key

In [1]:
# ========================================
# 配置 SiliconFlow API Key
# ========================================

SILICONFLOW_API_KEY = "sk-qdbrvnasyqcpqsvekchvkvrsuikujzfwgoltoytnegnlvdbk"  # 替换为你的 API Key

# 选择 VLM 模型
# 推荐: "Qwen/Qwen2.5-VL-32B-Instruct" (高性价比)
# 免费: "THUDM/GLM-4.1V-9B-Thinking" (完全免费但效果略差)
VLM_MODEL = "Qwen/Qwen2.5-VL-32B-Instruct"

# ========================================

import os
os.environ["SILICONFLOW_API_KEY"] = SILICONFLOW_API_KEY
print(f"✓ Using SiliconFlow VLM: {VLM_MODEL}")

✓ Using SiliconFlow VLM: Qwen/Qwen2.5-VL-32B-Instruct


## Step 1: 安装依赖

In [ ]:
import time
start = time.time()

# 安装基础依赖
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.49.0 accelerate safetensors
!pip install -q --no-deps git+https://github.com/deepseek-ai/Janus.git
!pip install -q attrdict sentencepiece
!pip install -q open-clip-torch timm einops peft bitsandbytes
!pip install -q tqdm Pillow matplotlib

# OpenAI SDK (用于调用 SiliconFlow API)
!pip install -q openai

print(f"\n✓ Installation complete: {time.time()-start:.1f}s")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 21.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 32.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 25.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 54.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 2: 测试 VLM API 连接

In [ ]:
from openai import OpenAI
import base64
from PIL import Image
import io

# 创建测试图像
test_img = Image.new('RGB', (256, 256), color='red')
buffer = io.BytesIO()
test_img.save(buffer, format='PNG')
img_base64 = base64.b64encode(buffer.getvalue()).decode()

# 测试 API
client = OpenAI(
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1"
)

response = client.chat.completions.create(
    model=VLM_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What color is this image? Reply in one word."},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{img_base64}"},
                },
            ],
        }
    ],
    max_tokens=100,
)

print(f"✓ VLM API 测试成功!")
print(f"Response: {response.choices[0].message.content}")

## Step 3: 检查 GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU found! Training will be very slow.")

## Step 4: 克隆项目代码

In [ ]:
import os

if not os.path.exists('T2I-RL-Project'):
    !git clone https://github.com/Inoriany/T2I-RL-Project.git
    print("✓ Repository cloned")
else:
    !cd T2I-RL-Project && git pull
    print("✓ Repository updated")

os.chdir('T2I-RL-Project')
print(f"Working directory: {os.getcwd()}")

## Step 5: 加载 Janus-Pro 模型 (4-bit 量化)

In [ ]:
import sys
sys.path.insert(0, 'src')

from models.generators import JanusProGenerator

print("Loading Janus-Pro-1B with 4-bit quantization...")
generator = JanusProGenerator(
    model_name_or_path="deepseek-ai/Janus-Pro-1B",
    device="cuda",
    load_in_4bit=True  # 节省显存
)
generator.load_model()

# 测试生成
test_prompt = "a red apple on a wooden table"
images = generator.generate(prompt=test_prompt)

print(f"✓ Generator loaded and tested")
display(images[0])

## Step 6: 创建 VLM Reward Model

In [ ]:
from models.reward_models import VLMRewardModel

# 创建 VLM Reward Model (使用 SiliconFlow API)
reward_model = VLMRewardModel(
    device="cuda",
    use_api=True,
    api_model=VLM_MODEL  # 使用 SiliconFlow 的模型
)
reward_model.load_model()

# 测试 reward 计算
test_images = generator.generate(prompt=test_prompt)
reward_output = reward_model.compute_reward(
    images=test_images,
    prompts=[test_prompt]
)

print(f"✓ VLM Reward Model ready")
print(f"Test reward: {reward_output.rewards.item():.4f}")
print(f"Details: {reward_output.details}")

## Step 7: 准备训练数据

In [ ]:
import json

def extract_prompt_texts(data):
    # 支持多种 schema:
    # 1) ["prompt a", "prompt b"]
    # 2) [{"prompt": "..."}, ...]
    # 3) {"prompts": [...]}
    # 4) {"prompts": {"cat": [...], ...}}
    if isinstance(data, list):
        if len(data) == 0:
            return []
        if isinstance(data[0], str):
            return data
        if isinstance(data[0], dict):
            return [d['prompt'] for d in data if isinstance(d, dict) and 'prompt' in d]

    if isinstance(data, dict):
        prompts = data.get('prompts', data)
        if isinstance(prompts, list):
            if len(prompts) == 0:
                return []
            if isinstance(prompts[0], str):
                return prompts
            if isinstance(prompts[0], dict):
                return [d['prompt'] for d in prompts if isinstance(d, dict) and 'prompt' in d]
        if isinstance(prompts, dict):
            flat = []
            for category, items in prompts.items():
                if isinstance(items, list):
                    if len(items) > 0 and isinstance(items[0], str):
                        flat.extend(items)
                    elif len(items) > 0 and isinstance(items[0], dict):
                        flat.extend([d['prompt'] for d in items if isinstance(d, dict) and 'prompt' in d])
            return flat

    raise ValueError('Unsupported prompt file format in train_prompts.json')

# 加载训练 prompts
with open('data/prompts/train_prompts.json', 'r') as f:
    raw_data = json.load(f)

all_prompts = extract_prompt_texts(raw_data)
print(f"Total prompts available: {len(all_prompts)}")

if len(all_prompts) == 0:
    raise ValueError('No prompts found in data/prompts/train_prompts.json')

# 快速训练: 最多用 50 个 prompts
NUM_PROMPTS = min(50, len(all_prompts))
train_prompts = all_prompts[:NUM_PROMPTS]

print(f"Using {NUM_PROMPTS} prompts for quick training")
print(f"\nExample prompts:")
for p in train_prompts[:5]:
    print(f"  - {p}")

## Step 8: 开始 GRPO 训练

**预计时间**: ~2-3 小时 (50 prompts, 3 epochs)

In [ ]:
from training.grpo_trainer import GRPOTrainer, GRPOConfig
from data.dataset import PromptDataset
from torch.utils.data import DataLoader
from datetime import datetime
import torch
import os

# 可选: 启用 LoRA（参数高效训练）
if not getattr(generator, 'lora_enabled', False):
    generator.enable_lora(lora_config={
        'r': 8,
        'lora_alpha': 16,
        'lora_dropout': 0.05,
    })

# 训练配置（与 GRPOTrainer 接口一致）
config = GRPOConfig(
    num_epochs=3,
    batch_size=1,
    learning_rate=1e-5,
    num_samples_per_prompt=2,   # REINFORCE-style scoring: fits on L4
    gradient_accumulation_steps=8,
    save_steps=50,
    logging_steps=10,
    output_dir='./outputs/grpo_siliconflow_quick',
    use_wandb=False,
    kl_coef=0.0,  # Colab/T4 稳定优先，先关 KL
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Dataloader
train_dataset = PromptDataset(train_prompts)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda x: {'prompt': [item['prompt'] for item in x]},
)

# 创建 trainer
trainer = GRPOTrainer(
    generator=generator,
    reward_model=reward_model,
    train_dataloader=train_dataloader,
    eval_dataloader=None,
    grpo_config=config,
)

# 自动从崩溃 checkpoint 恢复（如果存在）
resume_ckpt = './outputs/grpo_siliconflow_quick/crash_checkpoint'
resume_state = os.path.join(resume_ckpt, 'training_state.pt')
if os.path.exists(resume_state):
    print(f'Resuming from checkpoint: {resume_ckpt}')
    trainer.load_checkpoint(resume_ckpt)

    # Ensure trainable params exist after resume (PEFT can default to frozen)
    trainable = [p for p in trainer.generator.model.parameters() if p.requires_grad]
    print(f'Trainable params after resume: {len(trainable)}')
    if len(trainable) == 0:
        raise RuntimeError('No trainable parameters after resume. Re-run Step 5-8 from scratch.')
else:
    print('No crash checkpoint found, starting fresh training.')

print('Starting GRPO training with VLM reward...')
print(f'  Prompts: {len(train_prompts)}')
print(f'  Epochs: {config.num_epochs}')
print(f'  Batch size: {config.batch_size}')
print(f'  Samples per prompt: {config.num_samples_per_prompt}')
print(f'  Current epoch: {trainer.current_epoch}')
print(f'  Current global step: {trainer.global_step}')
print(f'  Total training steps (max): ~{len(train_dataloader) * config.num_epochs}')

In [ ]:
# 运行训练
if torch.cuda.is_available():
    torch.cuda.empty_cache()
start_time = datetime.now()
try:
    trainer.train()
except Exception as e:
    print(f'\nTraining failed: {type(e).__name__}: {e}')
    # 保存当前状态，便于恢复
    trainer.save_checkpoint('crash_checkpoint')
    raise
finally:
    elapsed = datetime.now() - start_time

training_history = {
    'global_step': int(trainer.global_step),
    'num_epochs': int(config.num_epochs),
    'num_prompts': int(len(train_prompts)),
    'elapsed_minutes': round(elapsed.total_seconds() / 60, 2),
}

print('\n' + '='*50)
print('Training Complete!')
print('='*50)
print(training_history)

## Step 9: 保存训练好的模型

In [ ]:
# 保存 checkpoint
trainer.save_checkpoint("final_checkpoint")
print("✓ Model checkpoint saved: ./outputs/grpo_siliconflow_quick/final_checkpoint/")

# 保存训练历史
import json
with open("./outputs/grpo_siliconflow_quick/training_history.json", "w") as f:
    json.dump(training_history, f, indent=2)
print("✓ Training history saved")

## Step 10: Before vs After 对比

In [ ]:
import matplotlib.pyplot as plt

# 测试 prompts
test_prompts = [
    "a red cat sitting on a blue chair",
    "two green apples and one yellow banana",
    "a small dog under a large table",
]

# 生成 After 图像 (训练后)
after_images = []
after_rewards = []

for prompt in test_prompts:
    img = generator.generate(prompt=prompt)[0]
    reward = reward_model.compute_reward([img], [prompt]).rewards.item()
    after_images.append(img)
    after_rewards.append(reward)

# 可视化
fig, axes = plt.subplots(1, len(test_prompts), figsize=(15, 5))

for i, (prompt, img, reward) in enumerate(zip(test_prompts, after_images, after_rewards)):
    axes[i].imshow(img)
    axes[i].set_title(f"Reward: {reward:.3f}\n{prompt[:30]}...", fontsize=10)
    axes[i].axis('off')

plt.suptitle("After Training (VLM Reward)", fontsize=14)
plt.tight_layout()
plt.savefig("./outputs/grpo_siliconflow_quick/after_training.png", dpi=150)
plt.show()

print("\nAverage reward after training:", sum(after_rewards) / len(after_rewards))

## Step 11: 下载结果

运行完成后，下载以下文件用于报告:
- `outputs/grpo_siliconflow_quick/final_checkpoint/` - 模型与优化器状态
- `outputs/grpo_siliconflow_quick/training_history.json` - 训练曲线
- `outputs/grpo_siliconflow_quick/after_training.png` - 生成图像

In [ ]:
# 打包结果
!zip -r results.zip outputs/grpo_siliconflow_quick/

from google.colab import files
files.download('results.zip')

print("✓ Results downloaded!")

---

## 下一步

1. **运行 Benchmark 评估** - 使用 `notebooks/benchmark_evaluation.ipynb`
2. **生成 Before/After Grid** - 使用 `notebooks/visualization.ipynb`
3. **Error Taxonomy 分析** - 使用 `notebooks/error_analysis.ipynb`